Setup de Spark 

In [ ]:
import os
import sys

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_CONF_DIR"] = os.path.expanduser("~/hadoop/etc/hadoop")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round as spark_round, lower, when
import matplotlib.pyplot as plt

spark = (
    SparkSession.builder
    .appName("Gaming Mental Health - Graphs")
    .master("local[*]")
    .getOrCreate()
)

spark

Chargement du dataset

In [ ]:
merged_path = "hdfs://localhost:9000/projet/gaming_mental_health/output/merged_dataset"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(merged_path)
)

print("Nombre de lignes :", df.count())
print("Nombre de colonnes :", len(df.columns))

# df.printSchema()
df.show(5)

## Gender distribution

In [ ]:
gender_distribution = (
    df.groupBy("gender")
    .agg(count("*").alias("count"))
    .orderBy("gender")
    .toPandas()
)

plt.figure(figsize=(6, 6))
plt.pie(
    gender_distribution["count"],
    labels=gender_distribution["gender"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Gender Distribution")
plt.show()

## 2. Game genre ranking

In [ ]:
game_distribution = (
    df.groupBy("game_genre")
    .agg(count("*").alias("count"))
    .orderBy(col("count").desc())
    .toPandas()
)

plt.figure(figsize=(9, 5))
plt.barh(game_distribution["game_genre"], game_distribution["count"])
plt.xlabel("Number of participants")
plt.ylabel("Game genre")
plt.title("Ranking of Game Genres")
plt.gca().invert_yaxis()
plt.show()

game_distribution

## 3. Average daily gaming hours by game genre

This chart compares the average number of daily gaming hours depending on the main game genre.

In [ ]:
from pyspark.sql.functions import col, avg, round as spark_round
import matplotlib.pyplot as plt

genre_hours_distribution = (
    df.groupBy("game_genre")
    .agg(spark_round(avg("daily_gaming_hours"), 2).alias("avg_daily_gaming_hours"))
    .orderBy(col("avg_daily_gaming_hours").desc())
    .toPandas()
)

plt.figure(figsize=(9, 5))
plt.barh(
    genre_hours_distribution["game_genre"],
    genre_hours_distribution["avg_daily_gaming_hours"]
)
plt.xlabel("Average daily gaming hours")
plt.ylabel("Game genre")
plt.title("Average Daily Gaming Hours by Game Genre")
plt.gca().invert_yaxis()
plt.show()

genre_hours_distribution

## 4. Physical symptoms related to gaming

This chart shows the percentage of participants reporting eye strain and back/neck pain.

In [ ]:
from pyspark.sql.functions import when, lower, avg, round as spark_round, col

symptoms_df = df.select(
    when(lower(col("eye_strain").cast("string")) == "true", 1).otherwise(0).alias("eye_strain_num"),
    when(lower(col("back_neck_pain").cast("string")) == "true", 1).otherwise(0).alias("back_neck_pain_num")
)

symptoms_distribution = symptoms_df.select(
    spark_round(avg("eye_strain_num") * 100, 2).alias("Eye strain"),
    spark_round(avg("back_neck_pain_num") * 100, 2).alias("Back/neck pain")
).toPandas()

symptoms_distribution = symptoms_distribution.melt(var_name="symptom", value_name="percentage")

plt.figure(figsize=(7, 5))
plt.bar(symptoms_distribution["symptom"], symptoms_distribution["percentage"])
plt.ylabel("Participants affected (%)")
plt.title("Physical Symptoms Related to Gaming")
plt.ylim(0, 100)
plt.show()

symptoms_distribution

## 5. Eye strain by gaming platform

This chart compares the percentage of participants reporting eye strain depending on their main gaming platform.

In [ ]:
platform_eye_distribution = (
    df.withColumn(
        "eye_strain_num",
        when(lower(col("eye_strain").cast("string")) == "true", 1).otherwise(0)
    )
    .groupBy("gaming_platform")
    .agg(spark_round(avg("eye_strain_num") * 100, 2).alias("eye_strain_percentage"))
    .orderBy(col("eye_strain_percentage").desc())
    .toPandas()
)

plt.figure(figsize=(8, 5))
plt.bar(
    platform_eye_distribution["gaming_platform"],
    platform_eye_distribution["eye_strain_percentage"]
)
plt.xlabel("Gaming platform")
plt.ylabel("Participants with eye strain (%)")
plt.title("Eye Strain by Gaming Platform")
plt.ylim(0, 100)
plt.show()

platform_eye_distribution

## 6. Back and neck pain by gaming platform

This chart compares back and neck pain reports depending on the main gaming platform.

In [ ]:
platform_back_pain_distribution = (
    df.withColumn(
        "back_neck_pain_num",
        when(lower(col("back_neck_pain").cast("string")) == "true", 1).otherwise(0)
    )
    .groupBy("gaming_platform")
    .agg(spark_round(avg("back_neck_pain_num") * 100, 2).alias("back_neck_pain_percentage"))
    .orderBy(col("back_neck_pain_percentage").desc())
    .toPandas()
)

plt.figure(figsize=(8, 5))
plt.bar(
    platform_back_pain_distribution["gaming_platform"],
    platform_back_pain_distribution["back_neck_pain_percentage"]
)
plt.xlabel("Gaming platform")
plt.ylabel("Participants with back/neck pain (%)")
plt.title("Back and Neck Pain by Gaming Platform")
plt.ylim(0, 100)
plt.show()

platform_back_pain_distribution

## 7. Mood state distribution

This chart shows the most common mood states reported by participants.

In [ ]:
from pyspark.sql.functions import count

mood_distribution = (
    df.groupBy("mood_state")
    .agg(count("*").alias("count"))
    .orderBy(col("count").desc())
    .toPandas()
)

plt.figure(figsize=(9, 5))
plt.barh(mood_distribution["mood_state"], mood_distribution["count"])
plt.xlabel("Number of participants")
plt.ylabel("Mood state")
plt.title("Mood State Distribution")
plt.gca().invert_yaxis()
plt.show()

mood_distribution

## 8. Gaming-related symptoms comparison

This chart compares the percentage of participants reporting withdrawal symptoms, loss of interest, and continued gaming despite problems.

In [ ]:
addiction_symptoms_df = df.select(
    when(lower(col("withdrawal_symptoms").cast("string")) == "true", 1).otherwise(0).alias("withdrawal_symptoms_num"),
    when(lower(col("loss_of_other_interests").cast("string")) == "true", 1).otherwise(0).alias("loss_of_other_interests_num"),
    when(lower(col("continued_despite_problems").cast("string")) == "true", 1).otherwise(0).alias("continued_despite_problems_num")
)

addiction_symptoms_distribution = addiction_symptoms_df.select(
    spark_round(avg("withdrawal_symptoms_num") * 100, 2).alias("Withdrawal symptoms"),
    spark_round(avg("loss_of_other_interests_num") * 100, 2).alias("Loss of other interests"),
    spark_round(avg("continued_despite_problems_num") * 100, 2).alias("Continued despite problems")
).toPandas()

addiction_symptoms_distribution = addiction_symptoms_distribution.melt(
    var_name="symptom",
    value_name="percentage"
)

plt.figure(figsize=(10, 5))
plt.bar(
    addiction_symptoms_distribution["symptom"],
    addiction_symptoms_distribution["percentage"]
)
plt.ylabel("Participants affected (%)")
plt.title("Gaming-related Symptoms Comparison")
plt.ylim(0, 100)
plt.xticks(rotation=20, ha="right")
plt.show()

addiction_symptoms_distribution